In [ ]:
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# go_df=pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/figure/circos-plot/segdup-feature/segdup_genes_hotspots.tsv", sep="\t", index_col=0)
# go_df=pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/figure/segdup-investigation/segdup_genes_hotspots_bpLevel_elemDecomposition.tsv", sep="\t", index_col=0)
# go_df=pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/figure/segdup-investigation/segdup_genes_hotspots_bpLevel_seqRemoved_top95percentile.tsv", sep="\t", index_col=0)
go_df=pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/figure/segdup-investigation/segdup_genes_hotspots_80percLength.tsv", sep="\t", index_col=0)

# background_df=pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/figure/segdup-investigation/segdup_genes_hotspots_bpLevel.tsv", sep="\t", index_col=0)
background_df = pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/figure/segdup-investigation/hifiasm_gene_segDup_overlapInfo_092525.tsv", sep="\t")

In [ ]:

print(background_df.shape)

In [ ]:
print(go_df.shape)

In [ ]:
search_terms = ["LARP7", "CCT4", "CCT7","MEPCE"]
pattern = "|".join(search_terms)

# This version handles missing values and is case-insensitive
filtered_go_df = go_df[
    go_df["gene"].str.contains(pattern, case=False, na=False)
]

filtered_go_df

In [ ]:
gene_list = go_df["gene"]
cleaned = gene_list.str.replace(r"-(?:DL|RL|L)\d+$", "", regex=True)
unique_genes=cleaned.unique()
print(len(unique_genes))
unique_genes

background_list = background_df["gene"]
cleaned = background_list.str.replace(r"-(?:DL|RL|L)\d+$", "", regex=True)
background_genes=cleaned.unique()
print(len(background_genes))

In [ ]:
import requests

# Download the latest human GO Biological Process GMT
url = "https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025"
response = requests.get(url)
with open("human_GO_bp_2025.gmt", "w") as f:
    f.write(response.text)

In [ ]:
# Filter out genes containing "LOC"
filtered_genes = unique_genes[~pd.Series(unique_genes).str.contains("LOC", na=False)]
filtered_genes = np.unique(filtered_genes)
print(len(filtered_genes))
filtered_genes


# Filter out genes containing "LOC"
background_genes = background_genes[~pd.Series(background_genes).str.contains("LOC", na=False)]
background_genes = np.unique(background_genes)
print(len(background_genes))

## Put this in metascape
filtered_genes

In [ ]:
list(filtered_genes)

In [ ]:
# Perform GO enrichment analysis
go_results  = gp.enrichr(
    gene_list=list(filtered_genes),  # Your converted gene list
    gene_sets="human_GO_bp_2025.gmt",     # Custom GMT file
    background=list(background_genes),  # Now this will be used!
    outdir=None
)

go_df = go_results.results
go_df.to_csv('go_enrichment_results_segdup_hotspot_seqRemoved_80percLength.csv', index=False)

# Plot top terms
plt.figure(figsize=(10, 8))
gp.dotplot(go_results.results, 
           title='GO Enrichment Analysis',
           cutoff=0.05,  # p-value cutoff
           top_term=10,  # show top 10 terms
           fontsize = 12,
           figsize=(10, 8),
           show_ring=True,
           cmap='viridis')
plt.savefig('go_enrichment_plot_segdup_hotspot_seqRemoved_80percLength.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import gseapy as gp

# Perform GO enrichment analysis
go_results = gp.enrichr(
    gene_list=list(filtered_genes),
    gene_sets="human_GO_bp_2025.gmt",
    background=list(background_genes),
    outdir=None
)

# Save results
go_df = go_results.results
# go_df.to_csv('go_enrichment_results_segdup_hotspot.csv', index=False)

# --- Publication-quality plot ---
plt.figure(figsize=(12, 6))

ax = gp.dotplot(
    go_results.results,
    title='GO Enrichment Analysis',
    cutoff=0.05,
    top_term=10,
    figsize=(10, 8),
    cmap='viridis_r',          # inverted colormap for contrast
    show_ring=True,
    marker='o',
    size=8,                    # increase circle size
    linewidth=1.2,             # thicker outline around circles
    edgecolor='black',         # black border for visibility
    alpha=0.9,                 # slightly transparent for overlapping
    fontsize=14,               # increase text size
)

# --- Further aesthetic adjustments ---
plt.title('GO Enrichment Analysis', fontsize=20, weight='bold', pad=20)
plt.xlabel('Combined Score', fontsize=16, labelpad=10)
plt.ylabel('', fontsize=16)
plt.xticks(fontsize=13)
plt.yticks(fontsize=13)
# plt.grid(False)

# Tight layout and high-res save
plt.tight_layout()
# plt.savefig('go_enrichment_plot_segdup_hotspot_pubready.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
ax.get_yticklabels()

In [ ]:
import textwrap

# Wrap long GO term labels to multiple lines
new_labels = []
for label in ax.get_yticklabels():
    text = label.get_text()
    wrapped = "\n".join(textwrap.wrap(text, width=30))  # wrap every 30 characters
    new_labels.append(wrapped)
ax.set_yticklabels(new_labels)

In [ ]:
go_results.results

In [ ]:
ax = gp.dotplot(
    go_results.results,
    title='GO Enrichment Analysis',
    cutoff=0.05,
    top_term=10,
    figsize=(10, 8),
    cmap='viridis_r',          # inverted colormap for contrast
    show_ring=True,
    marker='o',
    size=8,                    # increase circle size
    linewidth=1.2,             # thicker outline around circles
    edgecolor='black',         # black border for visibility
    alpha=0.9,                 # slightly transparent for overlapping
    fontsize=14,               # increase text size
)

In [ ]:

# Set global font sizes before plotting
plt.rcParams.update({
    'font.size': 20,
    'axes.titlesize': 22,
    'axes.labelsize': 18,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 13,
})

ax = gp.dotplot(
    go_results.results,
    title='GO Enrichment Analysis',
    cutoff=0.05,
    top_term=10,
    figsize=(10, 8.5),
    cmap='viridis_r',
    show_ring=True,
    marker='o',
    size=9,
    linewidth=1.2,
    edgecolor='black',
    alpha=0.9,
    fontsize=14,
)

# --- Add bullet points + wrap long GO term labels ---
new_labels = []
for label in ax.get_yticklabels():
    text = label.get_text()
    # Wrap every 30 characters and add a bullet point at the start
    wrapped = "\n".join(textwrap.wrap(text, width=30))
    wrapped = f"• {wrapped}"
    new_labels.append(wrapped)
ax.set_yticklabels(new_labels, fontsize=16, fontweight='bold')

# --- Further aesthetic adjustments ---
plt.title('GO Enrichment Analysis', fontsize=20, weight='bold', pad=20)
plt.xlabel('Combined Score', fontsize=16, labelpad=10)
plt.xticks(fontsize=13)
plt.yticks(fontsize=17, fontweight='bold')
# cbar = ax.collections[0].colorbar
# cbar.ax.tick_params(labelsize=14)  # <- controls the numeric tick font size

plt.tight_layout()
plt.savefig('go_enrichment_plot_segdup_hotspot_seqRemoved_80percLength_pubready.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
go_df[go_df["Term"].str.contains("Defense")]

In [ ]:
df_check=go_df.sort_values("Combined Score").tail(70)
df_check=

In [ ]:
go_df.sort_values("Combined Score")

In [ ]:
go_df[go_df["Term"].str.contains("Defense")]

In [ ]:
# plt.figure(figsize=(12, 8))
ax = gp.dotplot(
    go_results.results,
    title='GO Enrichment Analysis',
    cutoff=0.05,
    top_term=10,
    fontsize=16,          # Increase base font size
    figsize=(13, 10),
    show_ring=True,
    cmap='viridis'
)

# Make title, labels, and ticks more visible
# plt.title('GO Enrichment Analysis', fontsize=22, fontweight='bold',loc='left') 
plt.xlabel('Combined Score', fontsize=18, fontweight='bold')
plt.ylabel('', fontsize=14, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

# Adjust colorbar label
# cbar = plt.gcf().axes[-1]  # last axis is colorbar
# cbar.tick_params(labelsize=12)
# cbar.set_ylabel(r'$\log_{10}\frac{1}{FDR}$', fontsize=24, fontweight='bold')

plt.tight_layout()
plt.savefig('go_enrichment_plot_segdup_hotspot_bold.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
go_df

In [ ]:
go_df["Term"]

In [ ]:
go_df['GO_ID'] = go_df['Term'].str.extract(r'\((GO:\d+)\)')

In [ ]:
go_df

In [ ]:
go_df.to_csv("GO_terms_segdup_hotspot_regions.tsv", sep="\t")

In [ ]:
# Load genes
genes = pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/figure/segdup-investigation/hifiasm_gene_segDup_overlapInfo_092525.tsv", sep="\t")
genes["Start"] = genes["Start"].astype(int)
genes["End"] = genes["End"].astype(int)
genes["overlaps_dup"] = genes["overlaps_dup"].astype(bool)

In [ ]:
genes[genes["gene"].str.contains("LARP7")]

In [ ]:
# genes[genes["gene"]=="CCT7"]
genes[genes["gene"].str.contains("CCT7")]

In [ ]:
genes[genes["gene"].str.contains("CCT4")]